<a href="https://colab.research.google.com/github/andreslill/mosquito-season-suitability/blob/main/mosquito_suitability_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ERA5 Climate Suitability Pipeline for Seasonal Aedes Mosquito Activity


**Project:** When is mosquito season in your city?  
**Author:** Andrés Lill · March 2026  
**Data:** ERA5 monthly means, 1991–2020 (Copernicus CDS)  
**Output:** `mosquito_suitability.csv` (1,421 cities × 12 months)

## Overview

This pipeline was built to support an interactive public dashboard that compares seasonal climate suitability across cities worldwide.

Monthly climate suitability scores (0–1) are computed for two Aedes mosquito species across 1,421 cities worldwide.

**Species modelled:**
- *Ae. aegypti*: principal vector for dengue, Zika, chikungunya, and yellow fever;
  highly efficient transmission, tropical/subtropical distribution
  (Mordecai et al. 2017)
- *Ae. albopictus*: competent vector for the same arboviruses. Lower transmission
  efficiency for dengue and Zika but primary vector for chikungunya in temperate
  regions. Broader thermal tolerance and invasive range expansion into temperate
  regions including large parts of Europe
  (Bonizzoni et al. 2013, Tegar et al. 2026, ECDC Vector Maps)

**Suitability formula:**

```
Suitability Score (aegypti)    = TempScore × VPDScore
Suitability Score (albopictus) = TempScore × VPDScore × PhotoFactor*
```
*PhotoFactor: sigmoid transition (inflection = 23.5°, k = 0.5); continuous blend from tropical to temperate diapause behaviour

**Important:** Scores reflect *climate suitability*, not confirmed mosquito presence, disease risk, or actual population abundance.


## Notebook structure

| Section | Description |
|---|---|
| 1. Setup | Install dependencies, authenticate to Copernicus CDS |
| 2. Download ERA5 data | Download 30-year monthly means via CDS API |
| 3. Validate ERA5 files | Confirm expected variables and dimensions |
| 4. Compute suitability scores | Main pipeline: sample ERA5 → compute scores → export CSV |
| 5. Elevation enrichment | Fetch elevation via Open-Elevation API (first pass + retry) → export final CSV |

## Data sources

| Dataset | Source |
|---|---|
| Climate normals | ERA5 monthly means, Copernicus CDS (1991–2020) |
| City list | SimpleMaps World Cities Basic v1.901 (pop ≥ 500,000) |

## Model Parameters

| Model parameter | Source |
|---|---|
| Temperature suitability | Doeurk et al. 2025 (*Parasites & Vectors*) |
| VPD linearization | Schmidt et al. 2018 (*Parasites & Vectors*) |
| Photoperiod gate | Lacour et al. 2015 (*PLOS ONE*) |


## 1. Setup

### 1.1 Authenticate to Copernicus CDS

Reads your CDS API key from Colab Secrets (key name: `CDSAPI_KEY`) and writes the required `.cdsapirc` config file.

To get a CDS API key: register at [cds.climate.copernicus.eu](https://cds.climate.copernicus.eu), then find your key under your profile.

In [ ]:
from google.colab import userdata
import os, pathlib

# Read API key from Colab Secrets (never hardcode credentials)
key = userdata.get("CDSAPI_KEY")
assert key, "CDSAPI_KEY not found in Colab Secrets — add it under Tools > Secrets"

# Write the CDS API config file required by the cdsapi client
pathlib.Path("/root/.cdsapirc").write_text(
    "url: https://cds.climate.copernicus.eu/api\n"
    f"key: {key}\n"
)

### 1.2 Test CDS connection

Verify that the CDS client can connect successfully using the configured API key.

In [ ]:
!pip install cdsapi

import cdsapi
c = cdsapi.Client()
print("Connected")

## 2. Download ERA5 Data

Downloads ERA5 monthly averaged reanalysis for the 1991–2020 WMO reference period.

**Variables downloaded:**
- `2m_temperature` (K) used to derive temperature suitability
- `2m_dewpoint_temperature` (K) used with temperature to derive RH (%) and VPD (kPa)
- `total_precipitation` (m/day), included as contextual information only; not part of the suitability score formula

**Why monthly means?**  
We use a "typical year" climatology (30-year average per calendar month) instead of individual years, so that scores reflect long-term climate conditions rather than year-to-year variability.

**Note on units:**  
Temperature and dewpoint are returned in Kelvin (convert with `K - 273.15`).  
Precipitation is returned in m/day (convert to mm/month with `× 1000 × days_in_month`)

> *Climate data based on the 1991–2020 reference period. Recent warming trends may shift actual season windows beyond what this model shows.*



In [ ]:
import os
import cdsapi

# Update the paths below before running this notebook.
# Set your output path: Adjust to your local environment or Google Drive mount point
OUTPUT_PATH = "era5_monthly_1991_2020.nc"
os.makedirs(os.path.dirname(OUTPUT_PATH) or ".", exist_ok=True)

c = cdsapi.Client()
c.retrieve(
    "reanalysis-era5-single-levels-monthly-means",
    {
        "product_type": "monthly_averaged_reanalysis",
        "variable": [
            "2m_temperature",
            "2m_dewpoint_temperature",
            "total_precipitation",
        ],
        "year": [str(y) for y in range(1991, 2021)],
        "month": ["01","02","03","04","05","06","07","08","09","10","11","12"],
        "time": "00:00",
        "format": "netcdf",
    },
    OUTPUT_PATH,
)
print("Download complete.")

## 3. Validate ERA5 files

The CDS download returns separate NetCDF files for instantaneous variables (t2m, d2m) and accumulated variables (tp).
This short validation step confirms that the expected variables and dimensions are present before the main computation begins.

In [ ]:
import xarray as xr

# Update the paths below before running this notebook.
# Set your ERA5 extract directory: adjust to your local environment
EXTRACT_DIR = "./"  # e.g. "/content/drive/MyDrive/Mosquito Project/Data/"

# Load both NetCDF files
ds1 = xr.open_dataset(EXTRACT_DIR + "data_stream-moda_stepType-avgua.nc")  # t2m, d2m
ds2 = xr.open_dataset(EXTRACT_DIR + "data_stream-moda_stepType-avgad.nc")  # tp

print("=== ds1 (avgua): temperature + dewpoint ===")
print(ds1)
print("\n=== ds2 (avgad): precipitation ===")
print(ds2)

## 4. Compute Suitability Scores

Main pipeline: samples ERA5 climatology for each city, computes suitability scores, and exports a Tableau-ready CSV.
Intermediate model output is first saved as `mosquito_suitability_scores.csv`. Section 5 adds city level elevation data to produce the final file, `mosquito_suitability.csv`.

### Suitability model summary

**Temperature suitability (`TempScore`)**  
Triangular thermal performance curve: 0 at Tmin and Tmax, 1 at Topt, linear between.  
Parameters from Doeurk et al. 2025 (adult survival curve):

| Species | Tmin (°C) | Topt (°C) | Tmax (°C) |
|---|---|---|---|
| *Ae. aegypti* | 14.97 | 27.1 | 39.15 |
| *Ae. albopictus* | 11.02 | 24.5 | 38.07 |

**Desiccation stress (`VPDScore`)**  
Linearized vapour pressure deficit suitability (Schmidt et al. 2018):
- VPD ≤ 1.0 kPa: score = 1.0
- VPD ≥ 3.0 kPa: score = 0.0
- Linear between

VPD is derived from ERA5 temperature and dewpoint using the Magnus (Tetens) approximation.

**Photoperiod gate (PhotoFactor: *Ae. albopictus* only)**  
A sigmoid function (inflection = 23.5°, k = 0.5) weights the Lacour et al. 2015 
photoperiod thresholds (11.25 h / 13.5 h) continuously by latitude, producing a 
~5° transition zone around the tropics boundary. PhotoFactor approaches 1.0 near 
the equator and 0.0 at high latitudes in winter.

**Precipitation**  
Included as contextual information only. Does not contribute to the suitability score.


In [ ]:
"""
Mosquito Climate Suitability — ERA5 Pre-computation (1991–2020)
===============================================================

Purpose
-------
Generate a Tableau-ready CSV (city × month) with climate-based suitability
scores for Ae. aegypti and Ae. albopictus.

This model estimates *climate suitability for mosquito activity*, NOT disease
risk, NOT confirmed presence, and NOT mosquito abundance.

Climate baseline
----------------
ERA5 monthly means (Copernicus CDS) averaged over the 1991–2020 WMO
climatological reference period ("typical year" climatology).

> Climate data based on the 1991–2020 reference period. Recent warming
> trends may shift actual season windows beyond what this model shows.

Key methodological choices
--------------------------
1. Temperature suitability: triangular curve (0 at Tmin/Tmax, 1 at Topt).
   Parameters from Doeurk et al. 2025 (adult survival curves).

2. Desiccation stress: linearized VPD score (Schmidt et al. 2018):
   - VPD_score = 1.0  at VPD ≤ 1.0 kPa
   - VPD_score = 0.0  at VPD ≥ 3.0 kPa
   - linear between

3. Photoperiod (Ae. albopictus only, Lacour et al. 2015):
   A sigmoid function (inflection = 23.5°, k = 0.5) replaces the previous
   binary |lat| ≥ 23.5° cutoff. The temperate photoperiod thresholds from
   Lacour et al. 2015 (11.25 h / 13.5 h) are weighted continuously by latitude,
   producing a ~5° transition zone that eliminates edge-case artefacts near
   the tropics boundary (e.g. São Paulo vs. Rio de Janeiro).

4. Precipitation: included as contextual information only; not part of the
   suitability score formula. Classified relative to each city's own annual range
   (within-city quartiles) to avoid arbitrary global mm thresholds.

Outputs
-------
One row per city per month (12 rows per city) with:
- Climate normals (Temp, RH, VPD, precip, photoperiod)
- Component scores (temp_score_*, vpd_score, photo_factor_albopictus_temperate_only)
- Final suitability scores: suitability_score_aegypti, suitability_score_albopictus  [0–1]
- Contextual: precip_sum_mm (raw mm; relative context computed via within-city quartiles but not exported)
"""


import numpy as np
import pandas as pd
import xarray as xr
import math
from tqdm import tqdm

# ── CONFIG ────────────────────────────────────────────────────────
# Update the paths below before running this notebook.
# Adjust all paths to your local environment or Google Drive mount point
WORLDCITIES_CSV = "worldcities.csv"          # SimpleMaps World Cities Basic v1.901
EXTRACT_DIR     = "./"                        # directory containing the ERA5 NetCDF files
OUTPUT_CSV      = "mosquito_suitability_scores.csv"
MIN_POPULATION  = 500_000

# ── SPECIES TEMPERATURE PARAMETERS (Tmin, Topt, Tmax in °C) ──────
# Source: Doeurk et al. 2025 — adult survival curve parameters
TEMP_PARAMS = {
    "aegypti":    (14.97, 27.1,  39.15),
    "albopictus": (11.02, 24.5,  38.07),
}

# ── VPD THRESHOLDS (kPa) — linearized from Schmidt et al. 2018 ───
VPD_LOW  = 1.0   # VPD score = 1.0 at or below this threshold
VPD_HIGH = 3.0   # VPD score = 0.0 at or above this threshold

# ── PHOTOPERIOD THRESHOLDS (hours) — Lacour et al. 2015 ──────────
PHOTO_LOW  = 11.25   # minimum daylength for spring egg hatching
PHOTO_HIGH = 13.5    # critical photoperiod for diapause induction

# ── TROPICS BOUNDARY ──────────────────────────────────────────────
# Sigmoid inflection point for the photoperiod transition zone.
# Follows the astronomical tropics boundary (Tropic of Cancer/Capricorn).
TROPICS_LAT = 23.5


# ── HELPER FUNCTIONS ──────────────────────────────────────────────

def temp_score_triangle(temp_c: float, tmin: float, topt: float, tmax: float) -> float:
    """
    Triangular thermal performance curve.
    Returns 0 at Tmin/Tmax, 1 at Topt, linear between.
    """
    if temp_c <= tmin or temp_c >= tmax:
        return 0.0
    if temp_c <= topt:
        return (temp_c - tmin) / (topt - tmin)
    return (tmax - temp_c) / (tmax - topt)


def calc_vpd_kpa(temp_c: float, rh_pct: float) -> float:
    """
    Compute vapour pressure deficit (VPD) in kPa.
    Uses the Tetens (Magnus) approximation for saturation vapour pressure.
    """
    svp = 0.6108 * math.exp((17.27 * temp_c) / (temp_c + 237.3))
    return max(0.0, svp * (1.0 - rh_pct / 100.0))


def vpd_score(vpd_kpa: float) -> float:
    """
    Linearized VPD suitability score (Schmidt et al. 2018).
    1.0 below VPD_LOW, 0.0 above VPD_HIGH, linear between.
    """
    if vpd_kpa <= VPD_LOW:
        return 1.0
    if vpd_kpa >= VPD_HIGH:
        return 0.0
    return (VPD_HIGH - vpd_kpa) / (VPD_HIGH - VPD_LOW)


def photoperiod_hours(lat_deg: float, month: int) -> float:
    """
    Approximate daylength (hours) for a given latitude and month.
    Uses the 15th day of each month as a representative midpoint.
    """
    doy_mid = [15, 46, 74, 105, 135, 166, 196, 227, 258, 288, 319, 349]
    doy = doy_mid[month - 1]
    lat = math.radians(lat_deg)
    decl = math.radians(23.44) * math.sin(math.radians(360 / 365.0 * (doy - 81)))
    cos_ha = max(-1.0, min(1.0, -math.tan(lat) * math.tan(decl)))
    return round(2 * math.degrees(math.acos(cos_ha)) / 15.0, 2)


def albopictus_photo_factor(daylen_h: float, lat_deg: float,
                             k: float = 0.5, inflection: float = 23.5) -> float:
    """
    Ae. albopictus photoperiod gate — sigmoid transition across the tropics boundary.

    Replaces the binary |lat| ≥ 23.5° cutoff with a continuous sigmoid
    to avoid abrupt suitability jumps between adjacent cities (e.g. São Paulo
    vs. Rio de Janeiro). The inflection point at 23.5° follows the astronomical
    tropics boundary (Tropic of Cancer/Capricorn); k = 0.5 produces a ~5° 
    transition zone.    

    Within the transition zone, the temperate photoperiod thresholds from
    Lacour et al. 2015 (11.25 h / 13.5 h) are applied and then weighted by
    the sigmoid, so the photoperiod penalty scales continuously with latitude.
    """
    abs_lat = abs(lat_deg)

    # Sigmoid weight: 0.0 at equator, 1.0 at high latitudes
    sigmoid_weight = 1.0 / (1.0 + np.exp(-k * (abs_lat - inflection)))

    # Temperate photoperiod factor (Lacour et al. 2015)
    if daylen_h < PHOTO_LOW:
        temperate_factor = 0.0
    elif daylen_h < PHOTO_HIGH:
        temperate_factor = 0.5
    else:
        temperate_factor = 1.0

    # Blend: tropical (1.0) → temperate (temperate_factor) via sigmoid
    return round(1.0 - sigmoid_weight * (1.0 - temperate_factor), 4)

def days_in_month(month: int) -> int:
    """Days per calendar month (non-leap year). Used for mm/month conversion."""
    return [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31][month - 1]


def precip_context(monthly_precip_series: pd.Series) -> list:
    """
    Classify each month's precipitation relative to the city's own annual range.
    Uses within-city quartiles to avoid arbitrary global mm thresholds.
    Returns a list of labels: 'low', 'moderate', 'high', 'very_high'.
    """
    q25, q50, q75 = monthly_precip_series.quantile([0.25, 0.50, 0.75])
    cats = []
    for v in monthly_precip_series:
        if v <= q25:
            cats.append("low")
        elif v <= q50:
            cats.append("moderate")
        elif v <= q75:
            cats.append("high")
        else:
            cats.append("very_high")
    return cats


# ── LOAD ERA5 CLIMATOLOGY ─────────────────────────────────────────
ds1 = xr.open_dataset(EXTRACT_DIR + "data_stream-moda_stepType-avgua.nc")  # t2m, d2m (Kelvin)
ds2 = xr.open_dataset(EXTRACT_DIR + "data_stream-moda_stepType-avgad.nc")  # tp (m/day)

print("Computing 30-year monthly climatology (1991–2020)...")
t2m_clim = ds1["t2m"].groupby("valid_time.month").mean("valid_time")  # (12, lat, lon) Kelvin
d2m_clim = ds1["d2m"].groupby("valid_time.month").mean("valid_time")  # (12, lat, lon) Kelvin
tp_clim  = ds2["tp"].groupby("valid_time.month").mean("valid_time")   # (12, lat, lon) m/day


# ── LOAD AND FILTER CITIES ────────────────────────────────────────
cities = pd.read_csv(WORLDCITIES_CSV, usecols=["city", "country", "lat", "lng", "population"])
cities = cities.rename(columns={"lng": "lon"})
cities["population"] = pd.to_numeric(cities["population"], errors="coerce")
cities = cities.dropna(subset=["population", "lat", "lon"])
cities = cities[cities["population"] >= MIN_POPULATION]
cities = cities.drop_duplicates(subset=["city", "country"])
cities = cities.sort_values("population", ascending=False).reset_index(drop=True)
print(f"  {len(cities)} cities loaded (population ≥ {MIN_POPULATION:,})")


# ── MAIN SAMPLING LOOP ────────────────────────────────────────────
rows = []

for _, city in tqdm(cities.iterrows(), total=len(cities), desc="Cities"):
    lat, lon = float(city["lat"]), float(city["lon"])

    # ERA5 uses 0–360 longitude convention; convert negative (western) values
    lon360 = lon if lon >= 0 else lon + 360

    # Sample nearest ERA5 grid point for all 12 months at once
    t2m = t2m_clim.sel(latitude=lat, longitude=lon360, method="nearest").values  # (12,) Kelvin
    d2m = d2m_clim.sel(latitude=lat, longitude=lon360, method="nearest").values  # (12,) Kelvin
    tp  = tp_clim.sel(latitude=lat,  longitude=lon360, method="nearest").values  # (12,) m/day

    # Unit conversions
    temp_c = t2m - 273.15  # K → °C
    dewp_c = d2m - 273.15  # K → °C

    # Relative humidity (%) from temperature and dewpoint (Magnus approximation)
    rh = 100 * np.exp((17.27 * dewp_c) / (dewp_c + 237.3)) / \
               np.exp((17.27 * temp_c) / (temp_c + 237.3))
    rh = np.clip(rh, 0, 100)

    # Precipitation: m/day → mm/month (using exact days per month)
    precip_mm_month = np.array([float(tp[i]) * 1000 * days_in_month(i + 1) for i in range(12)])

    # Classify precipitation relative to city's own annual distribution
    precip_cats = precip_context(pd.Series(precip_mm_month))

    for i, month in enumerate(range(1, 13)):
        t   = float(temp_c[i])
        rh_ = float(rh[i])
        p   = float(precip_mm_month[i])

        vpd    = calc_vpd_kpa(t, rh_)
        vs     = vpd_score(vpd)
        daylen = photoperiod_hours(lat, month)

        ts_aeg = temp_score_triangle(t, *TEMP_PARAMS["aegypti"])
        ts_alb = temp_score_triangle(t, *TEMP_PARAMS["albopictus"])
        pf_alb = albopictus_photo_factor(daylen, lat)

        rows.append({
            # City metadata
            "city":                    city["city"],
            "country":                 city["country"],
            "lat":                     round(lat, 4),
            "lon":                     round(lon, 4),
            "population":              int(city["population"]),
            "month":                   month,

            # Climate normals (ERA5, 1991–2020)
            "temp_mean_c":             round(t, 2),
            "rh_mean_pct":             round(rh_, 1),
            "precip_sum_mm":           round(p, 1),
            "vpd_kpa":                 round(vpd, 4),
            "photoperiod_h":           daylen,

            # Suitability components
            "vpd_score":               round(vs, 4),
            "temp_score_aegypti":      round(ts_aeg, 4),
            "temp_score_albopictus":   round(ts_alb, 4),
            "photo_factor_albopictus_temperate_only": pf_alb,

            # Final suitability scores (0–1)
            "suitability_score_aegypti":      round(ts_aeg * vs, 4),
            "suitability_score_albopictus":   round(ts_alb * vs * pf_alb, 4),

        })
        # Note: precip_context label (low/moderate/high/very_high) is computed
        # but not exported — precip_sum_mm is included as raw contextual data instead.


# ── EXPORT ────────────────────────────────────────────────────────
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)

print(f"\nDone! {len(cities)} cities × 12 months = {len(df):,} rows")
print(f"Saved to: {OUTPUT_CSV}")
print("\nSample output (first 12 rows):")
print(df.head(12).to_string(index=False))


### 4.1 Inspect output

Quick sanity checks on the exported CSV.

In [ ]:
import pandas as pd

# Update the paths below before running this notebook.
# Set your output path: Adjust to your local environment or Google Drive mount point
OUTPUT_CSV = "mosquito_suitability_scores.csv"

df = pd.read_csv(OUTPUT_CSV)

print(f"Shape: {df.shape}  ({df['city'].nunique()} cities × 12 months)")
print(f"\nColumns:\n{list(df.columns)}")
print(f"\nSuitability score ranges:")
print(f"  aegypti:    {df['suitability_score_aegypti'].min():.3f} – {df['suitability_score_aegypti'].max():.3f}")
print(f"  albopictus: {df['suitability_score_albopictus'].min():.3f} – {df['suitability_score_albopictus'].max():.3f}")
print(f"\nSample — Hamburg, Germany:")
print(df[df['city'] == 'Hamburg'].to_string(index=False))

## 5. Elevation Enrichment

Adds elevation (metres above sea level) to each city using the
[Open-Elevation API](https://open-elevation.com).

Elevation is used as **contextual information** in the dashboard tooltips
and helps explain suitability patterns in high-altitude cities where
temperatures are too low for mosquito activity despite tropical latitudes
(e.g. Mexico City at 2,240 m, Addis Ababa at 2,355 m, Cochabamba at 2,558 m).

**Note:** Elevation does not contribute to the suitability score formula.

In [ ]:
# ── ELEVATION ENRICHMENT ──────────────────────────────────────────────────────
# Fetches city-level elevation (metres above sea level) from the Open-Elevation API.
# Sent in batches to respect API limits.
#
# Input:  mosquito_suitability_scores.csv  (1,421 cities × 12 months)
# Intermediate output: mosquito_suitability_and_elevation.csv (same structure + elevation_m)
# Final output after retry/cleanup: mosquito_suitability.csv
#
# Source: Open-Elevation API — https://open-elevation.com (open source, no key required)
# Note: Elevation is contextual only — it does not enter the suitability formula.

import requests
import pandas as pd
import time

# Load the suitability dataset
df = pd.read_csv('mosquito_suitability_scores.csv')

def get_elevations(df_input: pd.DataFrame, batch_size=100) -> pd.Series:
    """
    Fetch elevation for all unique city coordinates in batches using the Open-Elevation API.
    Returns a pandas Series of elevation values (metres) that can be assigned to the
    'elevation_m' column of the original DataFrame.
    """
    # Create a DataFrame of unique city coordinates
    unique_coords = df_input[['lat', 'lon']].drop_duplicates().reset_index(drop=True)
    coordinate_to_elevation = {}

    print(f"Fetching elevations for {len(unique_coords)} unique city coordinates in batches of {batch_size}...")

    for i in range(0, len(unique_coords), batch_size):
        batch = unique_coords.iloc[i:i+batch_size]
        locations = [
            {"latitude": row.lat, "longitude": row.lon}
            for _, row in batch.iterrows()
        ]
        response = None
        try:
            response = requests.post(
                "https://api.open-elevation.com/api/v1/lookup",
                json={"locations": locations},
                timeout=30 # Add a timeout to prevent hanging indefinitely
            )
            response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)

            results = response.json().get('results', [])
            if not results:
                print(f"Warning: No elevation results found in response for batch starting at index {i}.\nResponse text: {response.text}")
                # Assign None for each location in the batch
                for _, row in batch.iterrows():
                    coordinate_to_elevation[(row.lat, row.lon)] = None
            else:
                for j, row_data in enumerate(results):
                    original_row = batch.iloc[j]
                    coordinate_to_elevation[(original_row.lat, original_row.lon)] = row_data['elevation']

        except requests.exceptions.Timeout:
            print(f"Request timed out for batch starting at index {i}.")
            for _, row in batch.iterrows():
                coordinate_to_elevation[(row.lat, row.lon)] = None
        except requests.exceptions.RequestException as e:
            print(f"Error fetching elevation for batch starting at index {i}: {e}")
            if response is not None and response.status_code:
                print(f"HTTP Status Code: {response.status_code}")
            if response is not None:
                print(f"Response text: {response.text}")
            for _, row in batch.iterrows():
                coordinate_to_elevation[(row.lat, row.lon)] = None
        except ValueError as e:  # JSONDecodeError base class
             print(f"JSON Decode Error for batch starting at index {i}: {e}")
             if response is not None and response.status_code:
                 print(f"HTTP Status Code: {response.status_code}")
             if response is not None:
                 print(f"Response text: {response.text}")
             for _, row in batch.iterrows():
                 coordinate_to_elevation[(row.lat, row.lon)] = None

        # Introduce a small delay to prevent hitting rate limits
        time.sleep(0.1)

    # Now, map the fetched elevations back to the original DataFrame
    # Create a Series that can be directly assigned to a new column
    return df_input.apply(lambda row: coordinate_to_elevation.get((row.lat, row.lon)), axis=1)

# Fetch elevations and add to dataframe
df['elevation_m'] = get_elevations(df)

# Save intermediate file with elevation per city × month row
df.to_csv('mosquito_suitability_and_elevation.csv', index=False)

print("\nElevation enrichment complete. Check 'mosquito_suitability_and_elevation.csv' for results.")
print("Sample of DataFrame with elevation:")
print(df.head())

### 5.1 Merge elevation and inspect output

In [ ]:
import pandas as pd

# Load elevation data and deduplicate to one row per city
elevation = pd.read_csv('mosquito_suitability_and_elevation.csv')
elevation = elevation[['city', 'country', 'elevation_m']].drop_duplicates(
    subset=['city', 'country']
)

# Sanity checks
print(f"Elevation rows: {elevation.shape}")           # should be (1421, 3)
print(f"Duplicates: {elevation.duplicated(['city', 'country']).sum()}")  # should be 0

# Merge elevation into the suitability dataset
df = pd.read_csv('mosquito_suitability_scores.csv')
df = df.merge(elevation, on=['city', 'country'], how='left')

# Check for missing values
# Section 5.2 retries any gaps caused by API timeouts
 
missing = df['elevation_m'].isna().sum()
print(f"Missing elevation values after first pass: {missing}")

# Export final dataset
df.to_csv('mosquito_suitability.csv', index=False)
print(f"\nDone! Shape: {df.shape}")

# Quick preview
print("\nSample — high-altitude cities:")
sample = df[df['elevation_m'] > 2000][['city', 'country', 'elevation_m']].drop_duplicates().head(10)
print(sample.to_string(index=False))

### 5.2 Retry pass: fill remaining gaps

The Open-Elevation API occasionally times out for individual batches. 
This cell identifies any cities still missing elevation after the first pass 
and re-fetches them with smaller batches and longer delays between requests.

Expected result: 0 cities missing after retry.

In [ ]:
import requests
import pandas as pd
import time

# Load the suitability dataset with elevation from the first pass
df = pd.read_csv('mosquito_suitability.csv')

# Identify cities still missing elevation
missing_coords = (
    df[df['elevation_m'].isna()][['city', 'country', 'lat', 'lon']]
    .drop_duplicates()
    .reset_index(drop=True)
)
print(f"Cities missing elevation after first pass: {len(missing_coords)}")


def get_elevations_retry(coords_df: pd.DataFrame, batch_size: int = 50) -> dict:
    """
    Fetch elevation for a subset of coordinates using the Open-Elevation API.
    Uses smaller batches and longer inter-batch delays than the first pass
    to handle API rate limits and transient timeouts.

    Returns a dict mapping (lat, lon) -> elevation_m (or None on failure).
    """
    result = {}
    for i in range(0, len(coords_df), batch_size):
        batch = coords_df.iloc[i:i + batch_size]
        locations = [
            {"latitude": row.lat, "longitude": row.lon}
            for _, row in batch.iterrows()
        ]
        try:
            response = requests.post(
                "https://api.open-elevation.com/api/v1/lookup",
                json={"locations": locations},
                timeout=60,
            )
            response.raise_for_status()
            results = response.json()["results"]
            for j, row in enumerate(batch.itertuples()):
                result[(row.lat, row.lon)] = results[j]["elevation"]
            print(f"  OK: batch {i // batch_size + 1}, "
                  f"{min(i + batch_size, len(coords_df))}/{len(coords_df)}")
        except Exception as e:
            print(f"  Error in batch {i // batch_size + 1}: {e}")
            for _, row in batch.iterrows():
                result[(row.lat, row.lon)] = None
        time.sleep(2)
    return result


# Only run the retry if there are genuinely missing values
if len(missing_coords) > 0:
    elevation_map = get_elevations_retry(missing_coords)

    def fill_elevation(row):
        if pd.isna(row["elevation_m"]):
            return elevation_map.get((row["lat"], row["lon"]))
        return row["elevation_m"]

    df["elevation_m"] = df.apply(fill_elevation, axis=1)

    still_missing = df["elevation_m"].isna().sum() // 12
    print(f"\nCities still missing after retry: {still_missing}")

    # Overwrite with completed dataset
    df.to_csv("mosquito_suitability.csv", index=False)
    print("Saved: mosquito_suitability.csv")
else:
    print("No missing values — retry pass not needed.")


## 6. Special Interest Cities

Appends two cities that fall below the 500,000 population threshold but are epidemiologically significant:

- **Funchal, Madeira (Portugal):** Only EU territory outside Cyprus where *Ae. aegypti* is confirmed established (ECDC 2025).
- **Nicosia, Cyprus:** *Ae. aegypti* establishment confirmed (Simonin 2025; ECDC 2025).

These cities are excluded from the main pipeline due to the population filter but are added here manually so they appear in the dashboard with correct suitability scores.

A `special_interest` flag column is added to the full dataset (0 for all regular cities, 1 for these two) to allow targeted filtering in Tableau.


In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import math
import requests

# ── SPECIAL INTEREST CITIES ───────────────────────────────────────────────────
# Cities excluded from the main pipeline (population < 500,000) but
# epidemiologically significant due to confirmed Ae. aegypti establishment.

SPECIAL_CITIES = [
    {"city": "Funchal", "country": "Portugal", "lat": 32.6669, "lon": -16.9241, "population": 112000},
    {"city": "Nicosia", "country": "Cyprus",   "lat": 35.1667, "lon":  33.3667, "population": 330000},
]

# ── LOAD ERA5 CLIMATOLOGY ─────────────────────────────────────────────────────
# This section can be run standalone without CDS API access.
# Requirements: netcdf4 installed (!pip install netcdf4), ERA5 files present in EXTRACT_DIR.
EXTRACT_DIR = "/content/drive/MyDrive/Mosquito Project/Data/"
ds1 = xr.open_dataset(EXTRACT_DIR + "data_stream-moda_stepType-avgua.nc")
ds2 = xr.open_dataset(EXTRACT_DIR + "data_stream-moda_stepType-avgad.nc")

t2m_clim = ds1["t2m"].groupby("valid_time.month").mean("valid_time")
d2m_clim = ds1["d2m"].groupby("valid_time.month").mean("valid_time")
tp_clim  = ds2["tp"].groupby("valid_time.month").mean("valid_time")

# ── MODEL PARAMETERS ──────────────────────────────────────────────────────────
TEMP_PARAMS = {
    "aegypti":    (14.97, 27.1,  39.15),
    "albopictus": (11.02, 24.5,  38.07),
}
VPD_LOW, VPD_HIGH   = 1.0, 3.0
PHOTO_LOW, PHOTO_HIGH = 11.25, 13.5


def temp_score_triangle(temp_c, tmin, topt, tmax):
    if temp_c <= tmin or temp_c >= tmax: return 0.0
    if temp_c <= topt: return (temp_c - tmin) / (topt - tmin)
    return (tmax - temp_c) / (tmax - topt)

def calc_vpd_kpa(temp_c, rh_pct):
    svp = 0.6108 * math.exp((17.27 * temp_c) / (temp_c + 237.3))
    return max(0.0, svp * (1.0 - rh_pct / 100.0))

def vpd_score(vpd_kpa):
    if vpd_kpa <= VPD_LOW: return 1.0
    if vpd_kpa >= VPD_HIGH: return 0.0
    return (VPD_HIGH - vpd_kpa) / (VPD_HIGH - VPD_LOW)

def photoperiod_hours(lat_deg, month):
    doy_mid = [15, 46, 74, 105, 135, 166, 196, 227, 258, 288, 319, 349]
    doy = doy_mid[month - 1]
    lat = math.radians(lat_deg)
    decl = math.radians(23.44) * math.sin(math.radians(360 / 365.0 * (doy - 81)))
    cos_ha = max(-1.0, min(1.0, -math.tan(lat) * math.tan(decl)))
    return round(2 * math.degrees(math.acos(cos_ha)) / 15.0, 2)

def albopictus_photo_factor(daylen_h, lat_deg, k=0.5, inflection=23.5):
    abs_lat = abs(lat_deg)
    sigmoid_weight = 1.0 / (1.0 + np.exp(-k * (abs_lat - inflection)))
    if daylen_h < PHOTO_LOW: temperate_factor = 0.0
    elif daylen_h < PHOTO_HIGH: temperate_factor = 0.5
    else: temperate_factor = 1.0
    return round(1.0 - sigmoid_weight * (1.0 - temperate_factor), 4)

def days_in_month(month):
    return [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31][month - 1]


# ── COMPUTE SCORES ────────────────────────────────────────────────────────────
# Identical formula to Section 4 — no model changes, only different input cities.
rows = []

for city in SPECIAL_CITIES:
    lat, lon = city["lat"], city["lon"]
    lon360 = lon if lon >= 0 else lon + 360

    t2m = t2m_clim.sel(latitude=lat, longitude=lon360, method="nearest").values
    d2m = d2m_clim.sel(latitude=lat, longitude=lon360, method="nearest").values
    tp  = tp_clim.sel(latitude=lat,  longitude=lon360, method="nearest").values

    temp_c = t2m - 273.15
    dewp_c = d2m - 273.15
    rh = 100 * np.exp((17.27 * dewp_c) / (dewp_c + 237.3)) /                np.exp((17.27 * temp_c) / (temp_c + 237.3))
    rh = np.clip(rh, 0, 100)
    precip_mm_month = np.array([float(tp[i]) * 1000 * days_in_month(i + 1) for i in range(12)])

    for i, month in enumerate(range(1, 13)):
        t   = float(temp_c[i])
        rh_ = float(rh[i])
        p   = float(precip_mm_month[i])
        vpd    = calc_vpd_kpa(t, rh_)
        vs     = vpd_score(vpd)
        daylen = photoperiod_hours(lat, month)
        ts_aeg = temp_score_triangle(t, *TEMP_PARAMS["aegypti"])
        ts_alb = temp_score_triangle(t, *TEMP_PARAMS["albopictus"])
        pf_alb = albopictus_photo_factor(daylen, lat)

        rows.append({
            "city":                    city["city"],
            "country":                 city["country"],
            "lat":                     round(lat, 4),
            "lon":                     round(lon, 4),
            "population":              city["population"],
            "month":                   month,
            "temp_mean_c":             round(t, 2),
            "rh_mean_pct":             round(rh_, 1),
            "precip_sum_mm":           round(p, 1),
            "vpd_kpa":                 round(vpd, 4),
            "photoperiod_h":           daylen,
            "vpd_score":               round(vs, 4),
            "temp_score_aegypti":      round(ts_aeg, 4),
            "temp_score_albopictus":   round(ts_alb, 4),
            "photo_factor_albopictus_temperate_only": pf_alb,
            "suitability_score_aegypti":    round(ts_aeg * vs, 4),
            "suitability_score_albopictus": round(ts_alb * vs * pf_alb, 4),
        })

df_special = pd.DataFrame(rows)

# ── FETCH ELEVATION ───────────────────────────────────────────────────────────
def fetch_elevation(lat, lon):
    try:
        response = requests.post(
            "https://api.open-elevation.com/api/v1/lookup",
            json={"locations": [{"latitude": lat, "longitude": lon}]},
            timeout=30
        )
        return response.json()["results"][0]["elevation"]
    except Exception as e:
        print(f"Elevation fetch failed for ({lat}, {lon}): {e}")
        return None

elevation_map = {c["city"]: fetch_elevation(c["lat"], c["lon"]) for c in SPECIAL_CITIES}
df_special["elevation_m"] = df_special["city"].map(elevation_map)

print("Special cities — sample output:")
print(df_special[["city", "country", "month", "suitability_score_aegypti",
                   "suitability_score_albopictus", "elevation_m"]].to_string(index=False))

# ── APPEND TO MAIN DATASET ────────────────────────────────────────────────────
df_main = pd.read_csv("mosquito_suitability.csv")

# Add special_interest flag
df_main["special_interest"] = 0
df_special["special_interest"] = 1

df_out = pd.concat([df_main, df_special], ignore_index=True)
df_out.to_csv("mosquito_suitability.csv", index=False)

print(f"\nDone! {len(df_out)} rows total ({len(df_main)} original + {len(df_special)} special interest)")
print(f"Special interest cities added: {df_special['city'].unique().tolist()}")


In [ ]:
import numpy as np

# Unit Tests: Magnus Approximation and Sigmoid PhotoFactor
# This cell validates two core components of the suitability model:
# 1. The Magnus approximation used to derive VPD from temperature and dewpoint
# 2. The sigmoid-based PhotoFactor applied to *Ae. albopictus* outside the tropics

# 1. Magnus Approximation

def magnus_es(T):
    """
    Saturation vapour pressure in kPa (Magnus approximation).
    T: air temperature in °C
    """
    return 0.6108 * np.exp(17.27 * T / (T + 237.3))

def compute_vpd(T, Td):
    """
    Vapour Pressure Deficit (VPD) in kPa.
    T:  air temperature in °C
    Td: dewpoint temperature in °C
    VPD = es(T) - es(Td)
    """
    return magnus_es(T) - magnus_es(Td)

def test_magnus():
    # When T == Td, air is fully saturated → VPD must be exactly 0
    assert abs(compute_vpd(20, 20)) < 1e-10, \
        "FAIL: VPD at saturation (T == Td) is not zero"

    # Plausibility check: 25°C air temp, 0°C dewpoint → VPD ≈ 2.9 kPa
    vpd = compute_vpd(25, 0)
    assert 2.5 < vpd < 3.2, \
        f"FAIL: VPD outside expected range: {vpd:.3f} kPa"

    # VPD must be non-negative for all realistic T / Td combinations
    for T in range(10, 40):
        for Td in range(0, T):
            assert compute_vpd(T, Td) >= 0, \
                f"FAIL: negative VPD at T={T}, Td={Td}"

test_magnus()
print("Magnus approximation: all tests passed")


# -----------------------------------------------------------------------------

# 2. Sigmoid PhotoFactor (Ae. albopictus only)

# The PhotoFactor applies a continuous sigmoid weight by latitude,
# replacing a hard binary cutoff at 23.5°. It approaches 1.0 near the
# equator and 0.0 at high latitudes, with a ~5° transition zone around
# the tropics boundary. Parameters: inflection = 23.5°, k = 0.5.
# Photoperiod thresholds (11.25 h / 13.5 h) follow Lacour et al. 2015.
# The sigmoid transition is a modelling choice of this project.

def photo_factor(lat, k=0.5, inflection=23.5):
    """
    Sigmoid PhotoFactor as a function of latitude.
    lat:        latitude in decimal degrees (positive = North, negative = South)
    k:          steepness of the sigmoid transition
    inflection: latitude of the inflection point (default: 23.5° = tropics boundary)
    Returns a value between 0 and 1.
    """
    return 1 / (1 + np.exp(k * (abs(lat) - inflection)))

def test_sigmoid():
    # At the equator, PhotoFactor must be close to 1 (no photoperiod constraint)
    assert photo_factor(0) > 0.95, \
        f"FAIL: PhotoFactor at equator too low: {photo_factor(0):.3f}"

    # At the tropics boundary (23.5°), sigmoid inflection → factor must be ~0.5
    assert abs(photo_factor(23.5) - 0.5) < 0.01, \
        f"FAIL: PhotoFactor at 23.5° deviates from 0.5: {photo_factor(23.5):.3f}"

    # At 60°N (approx. Hamburg), photoperiod strongly limits winter activity
    assert photo_factor(60) < 0.1, \
        f"FAIL: PhotoFactor at 60° too high: {photo_factor(60):.3f}"

    # Symmetry: Northern and Southern hemisphere must return identical values
    assert abs(photo_factor(40) - photo_factor(-40)) < 1e-10, \
        "FAIL: PhotoFactor is not symmetric around the equator"

    # Monotonicity: PhotoFactor must decrease strictly with increasing latitude
    lats = [0, 10, 20, 30, 40, 50, 60]
    factors = [photo_factor(l) for l in lats]
    assert all(factors[i] > factors[i+1] for i in range(len(factors) - 1)), \
        "FAIL: PhotoFactor is not strictly decreasing with latitude"

test_sigmoid()
print("Sigmoid PhotoFactor: all tests passed")